In [12]:
import pandas as pd
import boto3
import json
from sqlalchemy import create_engine
from io import StringIO

# Load credentials
cred_path = r'C:\Users\Axxela\Desktop\CART\aws_credentials_sym.json'
with open(cred_path, 'r') as f:
    creds = json.load(f)

from urllib.parse import quote_plus

engine = create_engine(
    f"mysql+mysqlconnector://{creds['db_user']}:{quote_plus(creds['db_pass'])}@{creds['db_host']}/{creds['db_name']}"
)
s3 = boto3.client('s3', aws_access_key_id=creds['access_key'], aws_secret_access_key=creds['secret_key'])

MAPPING_CONFIG = {
    'AX': {'exchange': 'Itau', 'table': 'itau_trades_clearisk'},
    'EE': {'exchange': 'Marex', 'table': 'marex_trades_clearisk'},
    'GG': {'exchange': 'Marex', 'table': 'marex_trades_clearisk'},
    'OS': {'exchange': 'Marex', 'table': 'marex_trades_clearisk'},
    'KT': {'exchange': 'KGI', 'table': 'kgi_trades_clearisk'},
    'SYS': {'exchange': 'Stonex', 'table': 'stonex_trades_clearisk'}
}

def get_mapping(row):
    # We now look at 'clientnumber' for the prefix
    cn = str(row.get('clientnumber', '')).strip().upper()
    cg = str(row.get('clientgroup', '')).strip().upper()
    
    # 1. Check clientnumber for AX, EE, GG, OS, KT
    prefix = cn[:2]
    if prefix == 'AX': return MAPPING_CONFIG['AX']
    if prefix in ['EE', 'GG', 'OS']: return MAPPING_CONFIG['EE'] # Maps to Marex
    if prefix == 'KT': return MAPPING_CONFIG['KT']
    
    # 2. Check clientgroup for SYS (Stonex)
    if cg.startswith('SYS'): return MAPPING_CONFIG['SYS']
    
    return None

def process_file(bucket, key):
    response = s3.get_object(Bucket=bucket, Key=key)
    df = pd.read_csv(StringIO(response['Body'].read().decode('utf-8')), 
                     low_memory=False, 
                     on_bad_lines='skip')
    
    df.columns = [c.lower().strip() for c in df.columns]

    # Apply mapping using the whole row so we can check both columns
    df['map_data'] = df.apply(get_mapping, axis=1)
    valid_df = df[df['map_data'].notna()].copy()
    
    if valid_df.empty:
        # print(f"--- No matches in {key} based on clientnumber/group")
        return

    # Add Exchange and Table info
    valid_df['exchangecode'] = valid_df['map_data'].apply(lambda x: x['exchange'])
    valid_df['target_table'] = valid_df['map_data'].apply(lambda x: x['table'])
    
    # Construct CtrCode: sectyp-exchangecode-contractcode
    valid_df['ctrcode'] = (
        valid_df['sectyp'].astype(str) + '-' + 
        valid_df['exchangecode'].astype(str) + '-' + 
        valid_df['contractcode'].astype(str)
    )

    # Database Inspector to handle the 'datestr' column issue
    from sqlalchemy import inspect
    inspector = inspect(engine)
    
    for table_name, group_df in valid_df.groupby('target_table'):
        db_columns = [col['name'] for col in inspector.get_columns(table_name)]
        cols_to_upload = [c for c in group_df.columns if c in db_columns]
        
        upload_df = group_df[cols_to_upload]
        
        if not upload_df.empty:
            upload_df.to_sql(table_name, con=engine, if_exists='append', index=False)
            print(f"✅ Successfully uploaded {len(upload_df)} rows to {table_name}")
            
def sync_quarter(bucket_name, year):
    # Loop through Jan (01), Feb (02), and March (03)
    months = ['01', '02', '03']
    for month in months:
        print(f"\n--- STARTING MONTH: {month} ---")
        prefix = f"SYM/{year}/{month}/"
        paginator = s3.get_paginator('list_objects_v2')
        
        for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
            for obj in page.get('Contents', []):
                key = obj['Key']
                if "TRD_SYM_" in key and key.endswith(".csv") and "_ALGO_" not in key:
                    print(f"Reading: {key}")
                    process_file(bucket_name, key)

# Run for Jan-March 2026
sync_quarter('bi-cr3-prod', '2026')


--- STARTING MONTH: 01 ---
Reading: SYM/2026/01/01/TRD_SYM_20260101.csv
Reading: SYM/2026/01/02/TRD_SYM_20260102.csv


IntegrityError: (mysql.connector.errors.IntegrityError) 1062 (23000): Duplicate entry 'FUT-Itau-IO' for key 'itau_trades_clearisk.PRIMARY'
[SQL: INSERT INTO itau_trades_clearisk (clientgroup, sectyp, exchangecode, contractcode, price, ctrcode) VALUES (%(clientgroup)s, %(sectyp)s, %(exchangecode)s, %(contractcode)s, %(price)s, %(ctrcode)s)]
[parameters: [{'clientgroup': 'KOLJR', 'sectyp': 'FUT', 'exchangecode': 'Itau', 'contractcode': 'IO', 'price': 69346.49, 'ctrcode': 'FUT-Itau-IO'}, {'clientgroup': 'KOLJR', 'sectyp': 'FUT', 'exchangecode': 'Itau', 'contractcode': 'IO', 'price': 78322.28, 'ctrcode': 'FUT-Itau-IO'}, {'clientgroup': 'KOLJR', 'sectyp': 'FUT', 'exchangecode': 'Itau', 'contractcode': 'IO', 'price': 78322.28, 'ctrcode': 'FUT-Itau-IO'}, {'clientgroup': 'KOLJR', 'sectyp': 'FUT', 'exchangecode': 'Itau', 'contractcode': 'IO', 'price': 69337.39, 'ctrcode': 'FUT-Itau-IO'}, {'clientgroup': 'KOLJR', 'sectyp': 'FUT', 'exchangecode': 'Itau', 'contractcode': 'IO', 'price': 69364.69, '